In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
import tempfile
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- to_dataset_read_wrap ---
_CATE_TMP = tempfile.TemporaryDirectory()
_CATE_BASE = Path(_CATE_TMP.name)
_CATE_ORIGIN = _CATE_BASE / "lenta.csv"
pd.DataFrame({
    "gender": ["f", "m", "f"],
    "group": ["test", "control", "test"],
    "response_att": [1, 0, 1],
    "age": [25, None, 40],
}).to_csv(_CATE_ORIGIN, index=False)
_SAVED_DATASETS = []

def FIX_TO_DATASET_READ_WRAP_ONEHOT_ENCODING(df, cols):
    if isinstance(df, pl.DataFrame):
        return df.with_columns(pl.col(cols[0]).cast(pl.Utf8))
    return df.copy()

def FIX_TO_DATASET_READ_WRAP_PATH_LINKER(name):
    return SimpleNamespace(origin=_CATE_ORIGIN, base=_CATE_BASE / name)

class Dataset:
    def __init__(self, df, x_columns, y_columns, w_columns):
        self.df = df
        self.x_columns = x_columns
        self.y_columns = y_columns
        self.w_columns = w_columns
    def save(self, path):
        _SAVED_DATASETS.append((path, self.df, self.x_columns, self.y_columns, self.w_columns))

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_to_dataset_read_wrap(onehot_encoding, path_linker):
    def make_lenta() -> None:
        pathlinker = path_linker("lenta")
        df = pd.read_csv(pathlinker.origin)
        df = onehot_encoding(df, ["gender"])
        df = df.fillna(0)
        df["group"] = df["group"].apply(lambda x: {"test": 1, "control": 0}.get(x))
        y_columns = ["response_att"]
        w_columns = ["group"]
        x_columns = [column for column in df.columns if column not in y_columns + w_columns]
        ds = Dataset(df, x_columns, y_columns, w_columns)
        ds.save(pathlinker.base)
    return make_lenta

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_to_dataset_read_wrap(onehot_encoding, path_linker):
    def make_lenta() -> None:
        pathlinker = path_linker("lenta")
        df = pl.read_csv(pathlinker.origin)
        df = onehot_encoding(df, ["gender"])
        df = df.fill_null(0)
        df = df.with_columns(
            pl.col("group").replace({"test": 1, "control": 0}).alias("group")
        )
        y_columns = ["response_att"]
        w_columns = ["group"]
        x_columns = [column for column in df.columns if column not in y_columns + w_columns]
        ds = Dataset(df, x_columns, y_columns, w_columns)
        ds.save(pathlinker.base)
    return make_lenta

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: to_dataset_read_wrap ===

# L1 smoke – generated
try:
    _r = gen_to_dataset_read_wrap(FIX_TO_DATASET_READ_WRAP_ONEHOT_ENCODING, FIX_TO_DATASET_READ_WRAP_PATH_LINKER)
    print("✅ L1 smoke gen_to_dataset_read_wrap: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_to_dataset_read_wrap: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_to_dataset_read_wrap(FIX_TO_DATASET_READ_WRAP_ONEHOT_ENCODING, FIX_TO_DATASET_READ_WRAP_PATH_LINKER)
    print("✅ L1 smoke before_to_dataset_read_wrap: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_to_dataset_read_wrap: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence – execute the returned function and inspect saved state
def _run_to_dataset_read_wrap(factory):
    _SAVED_DATASETS.clear()
    _fn = factory(FIX_TO_DATASET_READ_WRAP_ONEHOT_ENCODING, FIX_TO_DATASET_READ_WRAP_PATH_LINKER)
    _result = _fn()
    return _result, _SAVED_DATASETS[-1]

try:
    _before_result, _before_saved = _run_to_dataset_read_wrap(before_to_dataset_read_wrap)
    _gen_result, _gen_saved = _run_to_dataset_read_wrap(gen_to_dataset_read_wrap)
    compare(_before_saved[1], _gen_saved[1], "to_dataset_read_wrap", check_row_order=True)
    if _before_result == _gen_result and _before_saved[2:] == _gen_saved[2:]:
        print("✅ L2 equivalence to_dataset_read_wrap return and metadata: MATCH")
    else:
        print(f"❌ L2 equivalence to_dataset_read_wrap return and metadata: MISMATCH — before={(_before_result, _before_saved[2:])}, gen={(_gen_result, _gen_saved[2:])}")
except Exception as _e:
    print(f"❌ L2 equivalence to_dataset_read_wrap: setup error — {type(_e).__name__}: {_e}")

# L3 edge – a schema-valid header-only CSV
try:
    _original = _CATE_ORIGIN.read_bytes()
    try:
        _CATE_ORIGIN.write_text("gender,group,response_att,age\n", encoding="utf-8")
        _before_result, _before_saved = _run_to_dataset_read_wrap(before_to_dataset_read_wrap)
        _gen_result, _gen_saved = _run_to_dataset_read_wrap(gen_to_dataset_read_wrap)
    finally:
        _CATE_ORIGIN.write_bytes(_original)
    compare(_before_saved[1], _gen_saved[1], "L3 edge to_dataset_read_wrap empty CSV", check_row_order=True)
    if _before_result == _gen_result and _before_saved[2:] == _gen_saved[2:]:
        print("✅ L3 edge to_dataset_read_wrap empty CSV metadata: MATCH")
    else:
        print(f"❌ L3 edge to_dataset_read_wrap empty CSV metadata: MISMATCH — before={(_before_result, _before_saved[2:])}, gen={(_gen_result, _gen_saved[2:])}")
except Exception as _e:
    print(f"❌ L3 edge to_dataset_read_wrap empty CSV: {type(_e).__name__}: {_e}")
